# 01 · Corpus lírico: MERGE Lyrics Complete traducido al español
**Fase 1 de la propuesta (Capítulo 4, Secciones 4.3.1 a 4.3.3).**

Este notebook:
1. Descarga MERGE Lyrics Complete desde Zenodo y verifica el archivo.
2. Carga letras, valencia, activación, cuadrante y la partición oficial 70-15-15.
3. Traduce las letras al español verso por verso con NLLB-200 (600M destilado).
4. Genera la muestra de 100 letras (25 por cuadrante) para el control de calidad manual.
5. Calcula cuántas letras superan los 512 tokens.

**Antes de empezar:** abre este notebook desde Jupyter (Anaconda), con el entorno `tesis` (ver GUIA.md).
La traducción tarda bastante, sobre todo sin GPU. Si cierras Jupyter a medias, vuelve a ejecutar todo: continúa donde quedó.

**`MODO_PRUEBA = True`** traduce solo 100 letras para comprobar que todo funciona. Cuando funcione, cámbialo a `False` y vuelve a ejecutar.

In [1]:
# Preparar rutas: la carpeta implementacion/ de tu computadora
import sys, os
from pathlib import Path

BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

CODIGO = BASE / 'codigo'
DATOS = BASE / 'datos'
RESULTADOS = BASE / 'resultados'
DATOS.mkdir(exist_ok=True); RESULTADOS.mkdir(exist_ok=True)
assert (CODIGO / 'datos.py').exists(), f'No encuentro {CODIGO}/datos.py. Abre este notebook desde la carpeta implementacion/notebooks'
sys.path.insert(0, str(CODIGO))

import torch
print('Carpeta del proyecto:', BASE)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no hay; se usará la CPU (más lento)')

Carpeta del proyecto: C:\Users\Usuario\IAT\TESIS
GPU: NVIDIA GeForce RTX 3050


In [2]:
import pandas as pd
m = pd.read_csv(f'{DATOS}/merge_en.csv')
print(m[['valencia', 'activacion']].describe().round(3))
print(m.groupby('cuadrante')[['valencia', 'activacion']].mean().round(3))

       valencia  activacion
count  2568.000    2568.000
mean      0.016      -0.048
std       0.462       0.281
min      -1.000      -1.000
25%      -0.412      -0.274
50%      -0.146       0.012
75%       0.475       0.142
max       1.000       0.950
           valencia  activacion
cuadrante                      
Q1            0.536       0.245
Q2           -0.392       0.127
Q3           -0.419      -0.269
Q4            0.407      -0.305


In [3]:
import hashlib, zipfile, urllib.request
import pandas as pd
from datos import *

MODO_PRUEBA = False        # True: solo 100 letras (prueba rápida) | False: corpus completo
SUFIJO = '_prueba' if MODO_PRUEBA else ''

## 1. Descargar MERGE Lyrics Complete (2.3 MB)

In [4]:
URL = 'https://zenodo.org/records/13939205/files/MERGE_Lyrics_Complete.zip?download=1'
MD5_ESPERADO = 'fae5bd689febeddc96595ac1843ffc77'   # publicado en Zenodo (versión 1.1.0)
ZIP = f'{DATOS}/MERGE_Lyrics_Complete.zip'
RAIZ_MERGE = f'{DATOS}/MERGE_Lyrics_Complete'

if not os.path.exists(ZIP):
    urllib.request.urlretrieve(URL, ZIP)
md5 = hashlib.md5(open(ZIP, 'rb').read()).hexdigest()
print('MD5:', md5, '✓' if md5 == MD5_ESPERADO else '✗ NO COINCIDE: vuelve a descargar')
if not os.path.exists(RAIZ_MERGE):
    zipfile.ZipFile(ZIP).extractall(RAIZ_MERGE)

MD5: fae5bd689febeddc96595ac1843ffc77 ✓


## 2. Explorar y cargar
`explorar_merge` muestra qué tablas trae el zip. Revisa que haya una con *Arousal/Valence*,
una con *Quadrant* y una carpeta de particiones `70_15_15`.

In [5]:
explorar_merge(RAIZ_MERGE)

MERGE_Lyrics_Complete\merge_lyrics_complete_av_values.csv  ->  (2568, 3)  columnas=['Song', 'Arousal', 'Valence']
MERGE_Lyrics_Complete\merge_lyrics_complete_metadata.csv  ->  (2568, 24)  columnas=['Song', 'Quadrant', 'AllMusic Id', 'AllMusic Extraction Date', 'Artist', 'Title', 'Relevance', 'Year', 'LowestYear', 'Duration', 'Moods', 'MoodsAll']
MERGE_Lyrics_Complete\tvt_dataframes\tvt_40_30_30\tvt_40_30_30_test_lyrics_complete.csv  ->  (768, 2)  columnas=['Song', 'Quadrant']
MERGE_Lyrics_Complete\tvt_dataframes\tvt_40_30_30\tvt_40_30_30_train_lyrics_complete.csv  ->  (1032, 2)  columnas=['Song', 'Quadrant']
MERGE_Lyrics_Complete\tvt_dataframes\tvt_40_30_30\tvt_40_30_30_validate_lyrics_complete.csv  ->  (768, 2)  columnas=['Song', 'Quadrant']
MERGE_Lyrics_Complete\tvt_dataframes\tvt_70_15_15\tvt_70_15_15_test_lyrics_complete.csv  ->  (384, 2)  columnas=['Song', 'Quadrant']
MERGE_Lyrics_Complete\tvt_dataframes\tvt_70_15_15\tvt_70_15_15_train_lyrics_complete.csv  ->  (1800, 2)  columnas=

In [6]:
merge = cargar_merge(RAIZ_MERGE, perfil='70_15_15')
# Si falla, pasa las rutas a mano, por ejemplo:
# merge = cargar_merge(RAIZ_MERGE, archivo_av='.../av_values.csv',
#                      archivos_particion={'train': '...train.csv', 'validacion': '...validate.csv', 'test': '...test.csv'})
print(len(merge), 'letras (esperado: 2568)')
resumen_merge(merge)     # esperado por cuadrante: Q1=600, Q2=710, Q3=621, Q4=637

V y A estaban en [0, 1]; se reescalaron a [-1, 1] con 2x - 1.
2568 letras (esperado: 2568)


particion,test,train,validacion,All
cuadrante,,,,
Q1,89,422,89,600
Q2,107,496,107,710
Q3,92,437,92,621
Q4,96,445,96,637
All,384,1800,384,2568


In [7]:
print(merge[['valencia', 'activacion']].describe())   # deberían estar en [-1, 1]
print(merge.letra_en.iloc[0][:500])
merge.to_csv(f'{DATOS}/merge_en.csv', index=False)

if MODO_PRUEBA:   # 100 letras manteniendo la proporción de particiones
    merge = merge.sample(n=100, random_state=42).reset_index(drop=True)
    print('MODO_PRUEBA: se traducen solo', len(merge), 'letras')

          valencia   activacion
count  2568.000000  2568.000000
mean      0.016446    -0.048371
std       0.461677     0.281427
min      -1.000000    -1.000000
25%      -0.412500    -0.273750
50%      -0.146250     0.012500
75%       0.475000     0.142500
max       1.000000     0.950000
Want to find out where the moon goes
When it leaves the western sky
And night dissolves again to morning
Azure turns to gold
Azure turns to gold

Gonna sleep with one eye open
Gonna keep the shades half drawn
Nearly silent dressed in shadows
Lines and colors fall
Lines and colors fall

Gonna watch her through the window
Just as I watched you before
Smile knows but just won't tell me
I just watch her go
I just watch her go

Now I know just where the moon goes
When it leaves the western sky
And ni


## 3. Traducción verso por verso con NLLB-200
Los versos repetidos (estribillos) se traducen una sola vez. Todo se guarda en
`datos/cache_traduccion.json`, así que si se corta no se pierde lo hecho (y lo traducido
en modo prueba se reutiliza en la corrida completa).

Sin GPU, baja `tam_lote` a 16 y usa `num_beams=1` si va demasiado lento (la traducción
sigue siendo determinista, solo un poco menos cuidada; anótalo en la tesis si lo usas).

In [8]:
versos = versos_unicos(merge.letra_en)
cache = traducir_versos(versos, ruta_cache=f'{DATOS}/cache_traduccion.json',
                        tam_lote=64 if torch.cuda.is_available() else 16, num_beams=4)

C:\Users\Usuario\anaconda3\envs\TESIS\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Versos únicos: 67152 | ya traducidos: 6200 | pendientes: 60952


Traduciendo:   0%|          | 0/953 [00:00<?, ?it/s]C:\Users\Usuario\anaconda3\envs\TESIS\lib\site-packages\transformers\models\m2m_100\modeling_m2m_100.py:552: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Traduciendo: 100%|██████████| 953/953 [39:23<00:00,  2.48s/it]


In [9]:
merge['letra_es'] = merge.letra_en.map(lambda l: aplicar_traduccion(l, cache))
merge['alertas'] = [', '.join(detectar_problemas_traduccion(o, t)) for o, t in zip(merge.letra_en, merge.letra_es)]
print('Letras con alguna alerta automática:', (merge.alertas != '').mean().round(3))
merge.to_csv(f'{DATOS}/merge_es{SUFIJO}.csv', index=False)

i = 0
for o, t in list(zip(merge.letra_en[i].split('\n'), merge.letra_es[i].split('\n')))[:15]:
    print(f'{o:<55} | {t}')

Letras con alguna alerta automática: 0.031
Want to find out where the moon goes                    | Quiero saber dónde va la luna
When it leaves the western sky                          | Cuando salga del cielo occidental
And night dissolves again to morning                    | Y la noche se disuelve de nuevo a la mañana
Azure turns to gold                                     | Azur se convierte en oro
Azure turns to gold                                     | Azur se convierte en oro
                                                        | 
Gonna sleep with one eye open                           | Voy a dormir con un ojo abierto
Gonna keep the shades half drawn                        | Voy a mantener los tonos medio dibujados
Nearly silent dressed in shadows                        | Casi en silencio vestido con sombras
Lines and colors fall                                   | Las líneas y los colores caen
Lines and colors fall                                   | Las líneas y los col

## 4. Muestra para el control de calidad (Sección 4.3.3)
Descarga `control_calidad_muestra.csv`, ábrelo en Excel o Google Sheets y califica cada letra.
La columna `alertas_automaticas` solo te ayuda a priorizar; la revisión es tuya.

In [10]:
muestra = muestra_control_calidad(merge, n_por_cuadrante=5 if MODO_PRUEBA else 25, semilla=42)
muestra.to_csv(f'{DATOS}/control_calidad_muestra{SUFIJO}.csv', index=False)
muestra.cuadrante.value_counts()

cuadrante
Q1    25
Q2    25
Q3    25
Q4    25
Name: count, dtype: int64

## 5. Longitud en tokens (Sección 4.3.5): proporción de letras que superan 512

In [11]:
from transformers import AutoTokenizer
from modelo import CODIFICADORES
tok = AutoTokenizer.from_pretrained(CODIFICADORES[0])
stats = proporcion_truncadas(merge.letra_es, tok)
print(stats)
pd.Series(stats).to_json(f'{RESULTADOS}/longitud_tokens_merge{SUFIJO}.json')

Token indices sequence length is longer than the specified maximum sequence length for this model (622 > 512). Running this sequence through the model will result in indexing errors


{'media_tokens': 379.4474299065421, 'mediana_tokens': 303.0, 'max_tokens': 1895, 'proporcion_mayor_512': 0.17873831775700935}
